# geo0221 – Coordinate Reference Systems in Code

**Unit:** geo0220 CRS & Projections · **Slides:** `geo0220_CRS_Projections` (pptx/pdf in the session folder) · **Data:** `data/Gauges_NRW_CRS_A.csv` … `_E.csv`

The same 254 river gauges of North Rhine-Westphalia are stored in five files, each in a different coordinate reference system (CRS). In QGIS (activity geo0222) you saw *where* the points land. Here you find out *how far off* they are.

You will learn to
1. **assign** a CRS (only the label) and **transform** to a CRS (new numbers),
2. **verify** a CRS guess with numbers instead of by eye,
3. **measure** distances only in a suitable projected CRS (in NRW: EPSG:25832),
4. avoid the **latitude/longitude order** trap,
5. hand a clean result to QGIS (GeoPackage in EPSG:25832).

*Data: LANUK NRW, OpenHygon gauge locations, Datenlizenz Deutschland – Zero – 2.0; coordinates converted for this exercise.*

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS, Geod, Transformer

DATA = Path("data")
PATTERN = "Gauges_NRW_CRS_*.csv"        # the mystery files
REF = "C"                               # file with a known CRS (degrees), used as reference
PAIR = ("2869500000200", "2763190000100")   # two station numbers (text) for §4: Goch (Niers), Bachum (Ruhr)

FILES = sorted(DATA.glob(PATTERN))
[p.name for p in FILES]

## 1. Five files, one set of gauges
### 1.1 Look before you read
**Predict:** which separator do the files use? Which columns hold the coordinates? Which file is different from the others?

In [ ]:
for p in FILES:
    lines = p.read_text(encoding="utf-8").splitlines()
    print(p.name)
    print("   ", lines[0])
    print("   ", lines[1])

### 1.2 Read all files the same way
The files name and order their coordinate columns differently. `read_points` hides that: it renames the coordinate columns to `x` (easting or longitude) and `y` (northing or latitude) and uses the station number as index, so the same gauge has the same index in every file. The station number is read as **text** (`dtype=str`): it is an identifier, nobody calculates with it, and leading zeros (DWD `00216`) must survive.

In [ ]:
X_NAMES = ["x", "east", "rechtswert", "longitude", "lon"]
Y_NAMES = ["y", "north", "hochwert", "latitude", "lat"]
ID_TYPES = {"station_no": str, "station_id": str}      # identifiers are text


def read_points(path):
    """CSV with points -> DataFrame with columns x, y; index = station number."""
    df = pd.read_csv(path, sep=None, engine="python", dtype=ID_TYPES)   # sep=None: pandas detects ; or ,
    df.columns = [c.lower() for c in df.columns]
    x = next(c for c in df.columns if c in X_NAMES)
    y = next(c for c in df.columns if c in Y_NAMES)
    key = "station_no" if "station_no" in df.columns else "station_id"
    return df.rename(columns={x: "x", y: "y"}).set_index(key)

In [ ]:
dfs = {p.stem.split("_")[-1]: read_points(p) for p in FILES}     # key = file letter
pd.concat({k: df[["name", "x", "y"]].head(2) for k, df in dfs.items()})

### 1.3 Value ranges: the strongest hint

In [ ]:
pd.DataFrame({k: df[["x", "y"]].agg(["min", "max"]).unstack() for k, df in dfs.items()}).T.round(1)

### 1.4 Your hypothesis
Write down one EPSG code per file. File C is done as an example. Candidates from the slides: EPSG:4326, EPSG:25832, EPSG:4647, EPSG:31466, EPSG:3857 (use [epsg.io](https://epsg.io) to compare).

Hints: degrees or metres? How many digits does the easting have? Does it start with *32* or with *2*? How large are the numbers compared with UTM?

In [ ]:
GUESS = {
    "C": "EPSG:4326",    # worked example: values around 51 and 7 are degrees (WGS 84 latitude/longitude)
    # YOUR CODE HERE
}
for k, epsg in sorted(GUESS.items()):
    print(k, epsg, "-", CRS(epsg).name)

## 2. Assign vs transform
A GeoDataFrame needs a geometry **and** a CRS. There are two different operations:

| | assign: `crs=…` / `set_crs` | transform: `to_crs` |
|---|---|---|
| what changes | only the label | the coordinate numbers |
| use it when | you know which CRS the numbers are in | you need the data in another CRS |
| in QGIS | *Layer Properties → Source → Assigned Coordinate Reference System* | *Export → Save Features As…* with another CRS |

**Watch out:** `points_from_xy(x, y)` expects **x = easting/longitude first**.

Worked example with file C:

In [ ]:
c = dfs[REF]                                   # the reference file in degrees
gc = gpd.GeoDataFrame(c, geometry=gpd.points_from_xy(c["x"], c["y"]), crs="EPSG:4326")   # assign
gc_utm = gc.to_crs("EPSG:25832")                                                           # transform
print(gc.loc[PAIR[0], "name"], gc.geometry.loc[PAIR[0]], "->", gc_utm.geometry.loc[PAIR[0]])

**Predict:** what happens if you *assign* EPSG:4326 to the transformed UTM numbers?

In [ ]:
wrong = gc_utm.set_crs("EPSG:4326", allow_override=True)    # only the label changes
print(wrong.geometry.loc[PAIR[0]], "-", wrong.crs.name)

Same numbers, now read as degrees: a longitude of about 300 000 "degrees" does not exist. **Assigning a CRS never repairs data.** Assign only the CRS the numbers really are in; transform when you need another one.

### 2.1 All files: assign your guess, transform to EPSG:25832

In [ ]:
def to_utm(df, epsg):
    """Assign the CRS `epsg` (our guess) and transform to EPSG:25832."""
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["x"], df["y"]), crs=epsg)
    return g.to_crs("EPSG:25832")


gdfs = {k: to_utm(dfs[k], epsg) for k, epsg in sorted(GUESS.items())}
{k: g.crs.to_epsg() for k, g in gdfs.items()}

## 3. Verify, don't guess
If a guess is right, the transformed points coincide with the transformed points of file C. The distance between "the same gauge" in two files tells you how right the guess is.

**Predict:** for which files do you expect 0.00 m?

In [ ]:
ref = gdfs[REF]
dev = pd.DataFrame({k: g.geometry.distance(ref.geometry.loc[g.index]) for k, g in gdfs.items()})
dev.agg(["mean", "max"]).round(3)          # metres

**Interpretation**
* **0.00 m:** the same points in the same datum (ETRS89), only another projection or notation. Your guess is right.
* **some decimetres, at most about 1 m:** also right, but the file uses an **old datum** (DHDN with the Bessel ellipsoid, see slides part 4). PROJ must change the datum, and that transformation has an accuracy of its own: a few metres officially for the 7-parameter transformation PROJ uses here (below 1 m in this data), about 0.1 m with the official BeTA2007 grid.
* **kilometres or `inf`:** wrong guess. Change `GUESS` in §1.4 and run §2.1 and §3 again.

> **Small print:** file C is WGS 84, the others are ETRS89. PROJ treats the two as equal (accuracy 1 m), so you see 0.00 m, although they differ by about 0.9 m in NRW today. That matters for precise GNSS work, not for this data.

## 4. Distances in the wrong CRS
### 4.1 Reference: geodesic distance on the ellipsoid

In [ ]:
geod = Geod(ellps="GRS80")
p, q = dfs[REF].loc[PAIR[0]], dfs[REF].loc[PAIR[1]]
_, _, d_ref = geod.inv(p.x, p.y, q.x, q.y)          # lon, lat of both points
print(f"{p['name']} -> {q['name']}: {d_ref / 1000:.3f} km (geodesic)")

### 4.2 The same distance with Pythagoras in each file's own numbers
**Predict first:** which files give a wrong distance, and by roughly which factor?

In [ ]:
rows = []
for k, df in dfs.items():
    p, q = df.loc[PAIR[0]], df.loc[PAIR[1]]
    d = np.hypot(p.x - q.x, p.y - q.y)                # Pythagoras in the file's own units
    rows.append({"file": k, "crs": GUESS.get(k), "distance": d, "ratio": d / d_ref})
pd.DataFrame(rows).round(4)

**Interpretation**
* **Degrees (C):** a "distance" of about 1.8 – in degrees, which is meaningless (a degree of longitude is shorter than a degree of latitude).
* **Web Mercator (E):** about **1.6 times too long** (scale 1/cos φ at 51.6° N). Fine for web map tiles, wrong for measuring.
* **UTM (A, B):** ratio ≈ 0.9998, about 16 cm per km too short. UTM shrinks distances on purpose near its central meridian 9° E (scale factor 0.9996) and less further west. Good for GIS work in NRW.
* **Gauss-Krüger (D):** ratio ≈ 1.0001, about 8 cm per km too long: the scale is exactly 1 on its central meridian 6° E and grows towards the east, where Bachum lies.

**Rule:** measure in a projected metric CRS (in NRW: EPSG:25832) or geodesically, never in degrees or Web Mercator.

## 5. The latitude/longitude trap
File C stores **latitude first**, like many GPS exports and like the original LANUK file. What happens if you take the columns in file order?

**Predict:** where on Earth will these points land?

In [ ]:
c_raw = pd.read_csv(DATA / "Gauges_NRW_CRS_C.csv", sep=";", dtype={"station_no": str})
swapped = gpd.GeoDataFrame(c_raw, crs="EPSG:4326",
                           geometry=gpd.points_from_xy(c_raw["latitude"], c_raw["longitude"]))
print("bounds (lon_min, lat_min, lon_max, lat_max):", swapped.total_bounds.round(1))
swapped.explore(tooltip="name", zoom_start=3)

Longitude ≈ 51° E, latitude ≈ 7° N: the coast of Somalia. **x = longitude/easting, y = latitude/northing.** In QGIS: *Delimited Text → X field = longitude, Y field = latitude*.

## 6. Hand over to QGIS
Save the reference points in EPSG:25832 as a GeoPackage and open it in QGIS together with your five CSV layers: every layer must lie on the GeoPackage points, and the gauges must sit on their rivers (OpenStreetMap or DTK WMS).

In [ ]:
out = DATA / "derived" / (PATTERN.split("_CRS")[0].lower() + "_25832.gpkg")
out.parent.mkdir(parents=True, exist_ok=True)
gdfs[REF].to_file(out, layer="points", driver="GPKG")
print("written:", out, "-", gdfs[REF].crs.name)

## 7. Wrap-up
1. Coordinates without a CRS are just two numbers: know the CRS and store it with the data.
2. **Assign** only the CRS the numbers really are in; **transform** when you need another CRS.
3. **Measure** in a projected metric CRS (NRW: EPSG:25832), never in degrees or Web Mercator.
4. **x = longitude/easting, y = latitude/northing.**

**Exit ticket:** one sentence: why did file D not give exactly 0.00 m, although your guess was right?

**Homework:** see the course plan in the README of your course repository.